# CALM 数据导入与算法对比测试

这个 notebook 用 `calm_dataset.CalmDataset` 生成 CALM-compatible 线性 SEM 数据，并在同一批数据上对比 kaifeng-jin/CALM、NOTEARS、`cd_A`、`cd_A_weakfaith`，可选加入 `cd_A_l1`。目标是先做导入/生成烟测，再给出可快速运行的小规模 benchmark。

In [1]:
import logging
import os
import sys
import time
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
# Prevent dual-libgomp (igraph + torch) SEGFAULT in multi-threaded Jupyter kernel
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "calm_dataset.py").exists() and (path / "coordinate_descent").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from MEC import find_v_structures, get_skeleton, is_in_markov_equiv_class
from calm_dataset import CalmDataset, weight_to_binary_adj
from coordinate_descent.coordinate0 import dag_coordinate_descent_l0
from coordinate_descent.cd_A_weakfaith import dag_coordinate_descent_l0_weakfaith

CALM_REPO_CANDIDATES = [
    Path(os.environ["CALM_REPO"]) if os.environ.get("CALM_REPO") else None,
    REPO_ROOT / "external" / "CALM",
    REPO_ROOT.parent / "CALM",
    Path(r"D:\tmp\CALM-inspect"),
]
CALM_REPO = next(
    (
        path.resolve()
        for path in CALM_REPO_CANDIDATES
        if path is not None and (path / "CALM.py").exists()
    ),
    None,
)

try:
    if CALM_REPO is None:
        raise FileNotFoundError(
            "Cannot find kaifeng-jin/CALM. Clone it and set CALM_REPO, e.g. "
            "git clone https://github.com/kaifeng-jin/CALM external/CALM"
        )
    if str(CALM_REPO) not in sys.path:
        sys.path.insert(0, str(CALM_REPO))
    import torch
    from CALM import calm as calm_algorithm
    HAS_CALM = True
    CALM_IMPORT_ERROR = None
except Exception as exc:
    HAS_CALM = False
    CALM_IMPORT_ERROR = exc

try:
    import igraph as _igraph
    HAS_IGRAPH = True
    IGRAPH_IMPORT_ERROR = None
except Exception as exc:
    HAS_IGRAPH = False
    IGRAPH_IMPORT_ERROR = exc

try:
    from coordinate_descent.cd_A_l1 import dag_coordinate_descent_l1
    HAS_CD_A_L1 = True
    CD_A_L1_IMPORT_ERROR = None
except Exception as exc:
    HAS_CD_A_L1 = False
    CD_A_L1_IMPORT_ERROR = exc

_previous_disable_level = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    from castle.algorithms import Notears
    HAS_NOTEARS = True
    NOTEARS_IMPORT_ERROR = None
except Exception as exc:
    HAS_NOTEARS = False
    NOTEARS_IMPORT_ERROR = exc
finally:
    logging.disable(_previous_disable_level)

try:
    from fges_compat import TetradSearch
    HAS_TETRAD_PC = True
    TETRAD_PC_IMPORT_ERROR = None
except Exception as exc:
    HAS_TETRAD_PC = False
    TETRAD_PC_IMPORT_ERROR = exc

try:
    TOOLBOX_ROOT = REPO_ROOT / "toolbox"
    if str(TOOLBOX_ROOT) not in sys.path:
        sys.path.append(str(TOOLBOX_ROOT))
    from cdt.metrics import SHD_CPDAG as cdt_shd_cpdag
except Exception:
    cdt_shd_cpdag = None

print(f"REPO_ROOT     : {REPO_ROOT}")
print(f"CalmDataset   : {'OK' if callable(CalmDataset) else 'MISSING'}")
print(f"python-igraph : {'OK' if HAS_IGRAPH else f'MISSING ({IGRAPH_IMPORT_ERROR})'}")
print(f"CALM repo     : {CALM_REPO if CALM_REPO is not None else 'MISSING'}")
print(f"CALM algorithm: {'OK' if HAS_CALM else f'MISSING ({CALM_IMPORT_ERROR})'}")
print(f"Tetrad PC     : {'OK' if HAS_TETRAD_PC else f'MISSING ({TETRAD_PC_IMPORT_ERROR})'}")
print(f"NOTEARS       : {'OK' if HAS_NOTEARS else f'MISSING ({NOTEARS_IMPORT_ERROR})'}")
print(f"cd_A          : {'OK' if callable(dag_coordinate_descent_l0) else 'MISSING'}")
print(f"cd_A_weakfaith: {'OK' if callable(dag_coordinate_descent_l0_weakfaith) else 'MISSING'}")
print(f"cd_A_l1       : {'OK' if HAS_CD_A_L1 else f'MISSING ({CD_A_L1_IMPORT_ERROR})'}")

/home/yin/DAG/experiments/notebooks/test/2026-05-13
REPO_ROOT     : /home/yin/DAG
CalmDataset   : OK
python-igraph : OK
CALM repo     : /home/yin/DAG/external/CALM
CALM algorithm: OK
Tetrad PC     : OK
NOTEARS       : OK
cd_A          : OK
cd_A_weakfaith: OK
cd_A_l1       : OK


Detecting 2 CUDA device(s).


## 1. CALM 导入和数据生成烟测

这一节只验证 `CalmDataset` 能正常实例化，并且返回的数据矩阵、加权 DAG、二值 DAG 形状一致。

In [2]:
try:
    smoke_ds = CalmDataset(
        n=200,
        d=8,
        graph_type="ER",
        degree=1.0,
        sem_type="gauss",
        noise_ratio=4.0,
        noise_scale_mode="variance",
        seed=123,
    )

    assert smoke_ds.X.shape == (200, 8)
    assert smoke_ds.B.shape == (8, 8)
    assert smoke_ds.B_bin.shape == (8, 8)
    assert np.isfinite(smoke_ds.X).all()

    CALM_READY = True
    CALM_SMOKE_ERROR = None
    print("CALM smoke test passed")
    print("X shape      :", smoke_ds.X.shape)
    print("true edges   :", int(weight_to_binary_adj(smoke_ds.B).sum()))
    print("noise scales :", np.round(smoke_ds.noise_scale, 3))
except Exception as exc:
    CALM_READY = False
    CALM_SMOKE_ERROR = exc
    print("CALM smoke test failed")
    print("error type:", type(exc).__name__)
    print("message   :", exc)
    raise RuntimeError(
        "CalmDataset generation failed. If the error mentions python-igraph, "
        "install igraph in the active notebook kernel and rerun this notebook."
    ) from exc

CALM smoke test passed
X shape      : (200, 8)
true edges   : 8
noise scales : [1.082 1.213 2.    1.    1.747 1.272 1.688 1.943]


## 2. Benchmark 配置

默认配置偏小，适合先确认流程能跑通。需要更稳定的结果时，可以增大 `trials`、`d_list`、`n_list` 和 `cd_T`。

In [3]:
CFG = {
    "trials": 1,
    "seed": 42,
    "d_list": [30],
    "n_list": [10000],
    "degree": 2.0,
    "graph_type": "ER",
    "sem_type": "gauss",
    "noise_ratios": [1.0, 16.0],
    "noise_scale_mode": "variance",
    "standardize": True,
    "cd_T": 100000,
    "cd_threshold": 0.05,
    "lambda_l0": 0.2,
    "lambda_l1": 0.05,
    "weakfaith_tau": 0.05,
    "weakfaith_screening": ["corr", "pcorr"],
    "weakfaith_combine": "union",
    "calm_lambda1": 0.005,
    "calm_alpha": 0.01,
    "calm_tau": 0.5,
    "calm_rho_init": 1e-5,
    "calm_rho_mult": 3.0,
    "calm_htol": 1e-8,
    "calm_subproblem_iter": 1000,
    "calm_standardize": False,
    "calm_device": "cpu",
    "include_cd_A_l1": True,
    "pc_alpha": 0.01,
    "pc_stable": True,
    "pc_depth": -1,
    "notears_lambda1": 0.1,
    "notears_loss_type": "l2",
    "notears_threshold": 0.1,
}

CFG

{'trials': 1,
 'seed': 42,
 'd_list': [30],
 'n_list': [10000],
 'degree': 2.0,
 'graph_type': 'ER',
 'sem_type': 'gauss',
 'noise_ratios': [1.0, 16.0],
 'noise_scale_mode': 'variance',
 'standardize': True,
 'cd_T': 100000,
 'cd_threshold': 0.05,
 'lambda_l0': 0.2,
 'lambda_l1': 0.05,
 'weakfaith_tau': 0.05,
 'weakfaith_screening': ['corr', 'pcorr'],
 'weakfaith_combine': 'union',
 'calm_lambda1': 0.005,
 'calm_alpha': 0.01,
 'calm_tau': 0.5,
 'calm_rho_init': 1e-05,
 'calm_rho_mult': 3.0,
 'calm_htol': 1e-08,
 'calm_subproblem_iter': 1000,
 'calm_standardize': False,
 'calm_device': 'cpu',
 'include_cd_A_l1': True,
 'pc_alpha': 0.01,
 'pc_stable': True,
 'pc_depth': -1,
 'notears_lambda1': 0.1,
 'notears_loss_type': 'l2',
 'notears_threshold': 0.1}

## 3. 评估函数和算法封装

In [4]:
def standardize_columns(X: np.ndarray) -> np.ndarray:
    scale = X.std(axis=0, keepdims=True)
    scale = np.where(scale > 0, scale, 1.0)
    return (X - X.mean(axis=0, keepdims=True)) / scale


def sample_covariance(X: np.ndarray) -> np.ndarray:
    X_centered = X - X.mean(axis=0, keepdims=True)
    return X_centered.T @ X_centered / X_centered.shape[0]




def tetrad_matrix_to_adj(df_result) -> np.ndarray:
    """Tetrad endpoint matrix -> 0/1 adjacency (ARROW=2, TAIL=3)."""
    mat = np.asarray(df_result.values if hasattr(df_result, "values") else df_result)
    d = mat.shape[0]
    G = np.zeros((d, d), dtype=int)
    for i in range(d):
        for j in range(i + 1, d):
            a, b = int(mat[i, j]), int(mat[j, i])
            if a == 2 and b == 3:
                G[i, j] = 1
            elif a == 3 and b == 2:
                G[j, i] = 1
            elif a == 3 and b == 3:
                G[i, j] = G[j, i] = 1
            elif a != 0 or b != 0:
                G[i, j] = G[j, i] = 1
    np.fill_diagonal(G, 0)
    return G

def shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    G_true = np.asarray(G_true, dtype=int)
    G_est = np.asarray(G_est, dtype=int)
    d = G_true.shape[0]
    dist = 0
    for i in range(d):
        for j in range(i + 1, d):
            if G_true[i, j] != G_est[i, j] or G_true[j, i] != G_est[j, i]:
                dist += 1
    return float(dist)


def precision_recall_f1(G_true: np.ndarray, G_est: np.ndarray):
    true_edge = G_true == 1
    pred_edge = G_est == 1
    tp = int(np.sum(true_edge & pred_edge))
    fp = int(np.sum((~true_edge) & pred_edge))
    fn = int(np.sum(true_edge & (~pred_edge)))
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2.0 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return float(precision), float(recall), float(f1)


def skeleton_precision_recall(G_true: np.ndarray, G_est: np.ndarray):
    skel_true = np.triu(get_skeleton(G_true), k=1).astype(bool)
    skel_est = np.triu(get_skeleton(G_est), k=1).astype(bool)
    tp = int(np.sum(skel_true & skel_est))
    fp = int(np.sum((~skel_true) & skel_est))
    fn = int(np.sum(skel_true & (~skel_est)))
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    return float(precision), float(recall)


def cpdag_shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    if cdt_shd_cpdag is not None:
        try:
            return float(cdt_shd_cpdag(G_true.astype(int), G_est.astype(int)))
        except Exception:
            pass

    skel_true = get_skeleton(G_true)
    skel_est = get_skeleton(G_est)
    skel_diff = int(np.sum(np.abs(skel_true - skel_est)) // 2)
    v_diff = len(find_v_structures(G_true).symmetric_difference(find_v_structures(G_est)))
    return float(skel_diff + v_diff)


def evaluate_graph(G_true: np.ndarray, G_est: np.ndarray) -> dict:
    G_est = np.asarray(G_est, dtype=int).copy()
    np.fill_diagonal(G_est, 0)
    directed_precision, directed_recall, directed_f1 = precision_recall_f1(G_true, G_est)
    skeleton_precision, skeleton_recall = skeleton_precision_recall(G_true, G_est)
    return {
        "mec_match": int(is_in_markov_equiv_class(G_true, G_est)),
        "shd": shd_score(G_true, G_est),
        "cpdag_shd": cpdag_shd_score(G_true, G_est),
        "directed_precision": directed_precision,
        "directed_recall": directed_recall,
        "directed_f1": directed_f1,
        "skeleton_precision": skeleton_precision,
        "skeleton_recall": skeleton_recall,
        "n_edges_est": int(G_est.sum()),
    }


def failure_result(message: str) -> dict:
    return {
        "status": "failed",
        "G_est": None,
        "objective": np.nan,
        "history_len": 0,
        "message": str(message),
    }

In [5]:
def run_notears(X: np.ndarray, S: np.ndarray, seed: int) -> dict:
    if not HAS_NOTEARS:
        return failure_result(f"NOTEARS unavailable: {NOTEARS_IMPORT_ERROR}")

    model = Notears(
        lambda1=CFG["notears_lambda1"],
        loss_type=CFG["notears_loss_type"],
        w_threshold=CFG["notears_threshold"],
    )
    previous_disable_level = logging.root.manager.disable
    logging.disable(logging.INFO)
    try:
        model.learn(X)
    finally:
        logging.disable(previous_disable_level)

    G_est = np.asarray(model.causal_matrix, dtype=int)
    np.fill_diagonal(G_est, 0)
    return {"status": "ok", "G_est": G_est, "objective": np.nan, "history_len": 0, "message": ""}


def run_calm(X: np.ndarray, S: np.ndarray, seed: int) -> dict:
    if not HAS_CALM:
        return failure_result(f"kaifeng-jin/CALM unavailable: {CALM_IMPORT_ERROR}")

    np.random.seed(seed)
    torch.manual_seed(seed)
    B_weighted = calm_algorithm(
        X,
        lambda1=CFG["calm_lambda1"],
        alpha=CFG["calm_alpha"],
        tau=CFG["calm_tau"],
        rho_init=CFG["calm_rho_init"],
        rho_mult=CFG["calm_rho_mult"],
        htol=CFG["calm_htol"],
        subproblem_iter=CFG["calm_subproblem_iter"],
        standardize=CFG["calm_standardize"],
        device=CFG["calm_device"],
    )
    G_est = (np.asarray(B_weighted) != 0).astype(int)
    np.fill_diagonal(G_est, 0)
    return {
        "status": "ok",
        "G_est": G_est,
        "objective": np.nan,
        "history_len": 0,
        "message": "",
    }


def run_pc(X: np.ndarray, S: np.ndarray, seed: int) -> dict:
    if not HAS_TETRAD_PC:
        return failure_result(f"Tetrad PC unavailable: {TETRAD_PC_IMPORT_ERROR}")

    df = pd.DataFrame(X, columns=[f"x{i}" for i in range(X.shape[1])]).astype("float64")
    search = TetradSearch(df)
    search.set_verbose(False)
    search.run_pc(
        alpha=CFG["pc_alpha"],
        stable=CFG["pc_stable"],
        depth=CFG["pc_depth"],
    )
    G_est = tetrad_matrix_to_adj(search.get_graph_to_matrix())
    return {"status": "ok", "G_est": G_est, "objective": np.nan, "history_len": 0, "message": ""}


def run_cd_A(X: np.ndarray, S: np.ndarray, seed: int) -> dict:
    A, G_est, obj, history = dag_coordinate_descent_l0(
        S,
        T=CFG["cd_T"],
        seed=seed,
        threshold=CFG["cd_threshold"],
        lambda_l0=CFG["lambda_l0"],
        return_history=True,
    )
    return {"status": "ok", "G_est": G_est, "objective": float(obj), "history_len": len(history), "message": ""}


def run_cd_A_weakfaith(X: np.ndarray, S: np.ndarray, seed: int) -> dict:
    A, G_est, obj, history = dag_coordinate_descent_l0_weakfaith(
        S,
        T=CFG["cd_T"],
        seed=seed,
        threshold=CFG["cd_threshold"],
        lambda_l0=CFG["lambda_l0"],
        return_history=True,
        faithfulness_tau=CFG["weakfaith_tau"],
        screening=CFG["weakfaith_screening"],
        combine=CFG["weakfaith_combine"],
    )
    return {"status": "ok", "G_est": G_est, "objective": float(obj), "history_len": len(history), "message": ""}


def run_cd_A_l1(X: np.ndarray, S: np.ndarray, seed: int) -> dict:
    if not HAS_CD_A_L1:
        return failure_result(f"cd_A_l1 unavailable: {CD_A_L1_IMPORT_ERROR}")

    A, G_est, obj, history = dag_coordinate_descent_l1(
        S,
        T=CFG["cd_T"],
        seed=seed,
        threshold=CFG["cd_threshold"],
        lambda_l1=CFG["lambda_l1"],
        return_history=True,
    )
    return {"status": "ok", "G_est": G_est, "objective": float(obj), "history_len": len(history), "message": ""}


ALGORITHMS = {
    "calm": run_calm,
    "pc_fisher_z": run_pc,
    "notears": run_notears,
    "cd_A": run_cd_A,
    "cd_A_weakfaith": run_cd_A_weakfaith,
}
if CFG["include_cd_A_l1"]:
    ALGORITHMS["cd_A_l1"] = run_cd_A_l1

list(ALGORITHMS)

['calm', 'pc_fisher_z', 'notears', 'cd_A', 'cd_A_weakfaith', 'cd_A_l1']

## 4. 运行对比

In [ ]:
if not globals().get("CALM_READY", False):
    raise RuntimeError(f"CALM data generation is not ready: {CALM_SMOKE_ERROR}")

rng = np.random.default_rng(CFG["seed"])
rows = []
last_graphs = {}

for d in CFG["d_list"]:
    for n_samples in CFG["n_list"]:
        trial_seeds = rng.integers(0, 10**9, size=CFG["trials"])
        for trial_id, data_seed_raw in enumerate(trial_seeds, start=1):
            data_seed = int(data_seed_raw)
            for noise_ratio in CFG["noise_ratios"]:
                dataset = CalmDataset(
                    n=n_samples,
                    d=d,
                    graph_type=CFG["graph_type"],
                    degree=CFG["degree"],
                    sem_type=CFG["sem_type"],
                    noise_ratio=noise_ratio,
                    noise_scale_mode=CFG["noise_scale_mode"],
                    seed=data_seed,
                )
                X = standardize_columns(dataset.X) if CFG["standardize"] else dataset.X
                S = sample_covariance(X)
                G_true = weight_to_binary_adj(dataset.B)
                last_graphs["true"] = G_true

                for alg_idx, (algorithm, runner) in enumerate(ALGORITHMS.items()):
                    alg_seed = data_seed + 1009 * (alg_idx + 1)
                    t0 = time.perf_counter()
                    try:
                        result = runner(X, S, alg_seed)
                    except Exception as exc:
                        result = failure_result(exc)
                    runtime_sec = time.perf_counter() - t0

                    base = {
                        "noise_ratio": noise_ratio,
                        "d": d,
                        "n_samples": n_samples,
                        "degree": CFG["degree"],
                        "trial_id": trial_id,
                        "data_seed": data_seed,
                        "algorithm": algorithm,
                        "status": result["status"],
                        "n_edges_true": int(G_true.sum()),
                        "noise_scale_min": float(np.min(dataset.noise_scale)),
                        "noise_scale_max": float(np.max(dataset.noise_scale)),
                        "runtime_sec": float(runtime_sec),
                        "objective": result["objective"],
                        "history_len": result["history_len"],
                        "message": result["message"],
                    }

                    if result["status"] == "ok":
                        metrics = evaluate_graph(G_true, result["G_est"])
                        row = {**base, **metrics}
                        last_graphs[algorithm] = result["G_est"]
                        print(
                            f"[{algorithm}] ratio={noise_ratio:g} d={d} n={n_samples} "
                            f"trial={trial_id} shd={row['shd']:.0f} "
                            f"cpdag_shd={row['cpdag_shd']:.0f} "
                            f"f1={row['directed_f1']:.3f} rt={runtime_sec:.2f}s"
                        )
                    else:
                        row = {
                            **base,
                            "mec_match": 0,
                            "shd": np.nan,
                            "cpdag_shd": np.nan,
                            "directed_precision": np.nan,
                            "directed_recall": np.nan,
                            "directed_f1": np.nan,
                            "skeleton_precision": np.nan,
                            "skeleton_recall": np.nan,
                            "n_edges_est": np.nan,
                        }
                        print(f"[SKIP] {algorithm}: {result['message']}")
                    rows.append(row)

df_trials = pd.DataFrame(rows)
display(df_trials)

[calm] ratio=1 d=30 n=10000 trial=1 shd=54 cpdag_shd=122 f1=0.219 rt=46.60s


May 15, 2026 10:04:07 AM java.util.prefs.FileSystemPreferences$6 run


Error computing BIC: This graph has a cycle.
[pc_fisher_z] ratio=1 d=30 n=10000 trial=1 shd=65 cpdag_shd=150 f1=0.364 rt=2.69s


## 5. 汇总和可视化

In [ ]:
ok = df_trials[df_trials["status"] == "ok"].copy()
if ok.empty:
    print("No successful algorithm runs.")
else:
    df_summary = (
        ok.groupby(["noise_ratio", "d", "n_samples", "degree", "algorithm"], as_index=False)
        .agg(
            trials=("trial_id", "count"),
            mec_match_mean=("mec_match", "mean"),
            shd_mean=("shd", "mean"),
            shd_std=("shd", "std"),
            cpdag_shd_mean=("cpdag_shd", "mean"),
            directed_f1_mean=("directed_f1", "mean"),
            skeleton_precision_mean=("skeleton_precision", "mean"),
            skeleton_recall_mean=("skeleton_recall", "mean"),
            n_edges_true_mean=("n_edges_true", "mean"),
            n_edges_est_mean=("n_edges_est", "mean"),
            runtime_sec_mean=("runtime_sec", "mean"),
            runtime_sec_std=("runtime_sec", "std"),
        )
        .sort_values(["noise_ratio", "d", "n_samples", "shd_mean", "runtime_sec_mean"])
        .reset_index(drop=True)
    )

    failures = (
        df_trials[df_trials["status"] != "ok"]
        .groupby(["noise_ratio", "d", "n_samples", "degree", "algorithm"], as_index=False)
        .size()
        .rename(columns={"size": "failures"})
    )
    df_summary = df_summary.merge(
        failures,
        on=["noise_ratio", "d", "n_samples", "degree", "algorithm"],
        how="left",
    )
    df_summary["failures"] = df_summary["failures"].fillna(0).astype(int)
    display(df_summary)

,noise_ratio,d,n_samples,degree,algorithm,trials,mec_match_mean,shd_mean,shd_std,cpdag_shd_mean,directed_f1_mean,skeleton_precision_mean,skeleton_recall_mean,n_edges_true_mean,n_edges_est_mean,runtime_sec_mean,runtime_sec_std,failures
0,1.0,30,10000,2.0,calm,2,0.0,39.0,22.627417,87.5,0.490620,0.882353,0.433333,60.0,28.0,110.474933,14.830819,0
1,1.0,30,10000,2.0,notears,2,0.0,64.0,9.899495,123.5,0.276991,0.619623,0.533333,60.0,51.5,574.741005,268.166630,0
2,1.0,30,10000,2.0,cd_A_weakfaith,2,0.0,71.5,10.606602,170.5,0.334944,0.471186,0.466667,60.0,59.5,30.710277,0.396152,0
3,1.0,30,10000,2.0,cd_A,2,0.0,73.5,0.707107,164.5,0.295400,0.445292,0.408333,60.0,55.0,32.619376,0.755097,0
4,1.0,30,10000,2.0,cd_A_l1,2,0.0,168.5,9.192388,490.0,0.216482,0.285004,0.900000,60.0,189.5,47.210755,3.604184,0
5,16.0,30,10000,2.0,calm,2,0.0,37.5,13.435029,90.0,0.525904,0.970588,0.408333,60.0,25.0,112.482018,18.963391,0
6,16.0,30,10000,2.0,notears,2,0.0,66.0,5.656854,129.0,0.261039,0.581111,0.458333,60.0,47.5,514.435526,73.341748,0
7,16.0,30,10000,2.0,cd_A,2,0.0,71.0,4.242641,159.5,0.277931,0.457143,0.366667,60.0,48.0,31.803306,3.161484,0
8,16.0,30,10000,2.0,cd_A_weakfaith,2,0.0,76.0,7.071068,164.0,0.250337,0.449318,0.416667,60.0,55.5,30.212873,1.892419,0
9,16.0,30,10000,2.0,cd_A_l1,2,0.0,165.0,15.556349,511.5,0.221541,0.286788,0.883333,60.0,185.0,46.355988,4.384784,0


In [ ]:
if not ok.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for algorithm, part in ok.groupby("algorithm"):
        curve = part.groupby("noise_ratio", as_index=False).agg(
            shd_mean=("shd", "mean"),
            runtime_sec_mean=("runtime_sec", "mean"),
        )
        axes[0].plot(curve["noise_ratio"], curve["shd_mean"], marker="o", label=algorithm)
        axes[1].plot(curve["noise_ratio"], curve["runtime_sec_mean"], marker="o", label=algorithm)

    axes[0].set_title("SHD vs noise_ratio")
    axes[0].set_xlabel("noise_ratio")
    axes[0].set_ylabel("mean SHD")
    axes[1].set_title("Runtime vs noise_ratio")
    axes[1].set_xlabel("noise_ratio")
    axes[1].set_ylabel("mean seconds")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()

## 6. 最后一次 trial 的图结构矩阵

In [ ]:
if last_graphs:
    names = list(last_graphs)
    fig, axes = plt.subplots(1, len(names), figsize=(3.2 * len(names), 3.2))
    if len(names) == 1:
        axes = [axes]
    for ax, name in zip(axes, names):
        ax.imshow(last_graphs[name], cmap="Greys", vmin=0, vmax=1)
        ax.set_title(name)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()